# 04 — RAG ATM end-to-end

## Retrieval-Augmented Generation: unir recuperación + LLM

Ya hemos construido por separado:

```text
01_embeddings_semantic_search
    → retrieve(question)

03_llm_application_basics
    → ask_atm_llm(question, context)
```

Ahora los conectaremos:

```text
                 CORPUS ATM
                     │
              embeddings
                     │
                     ▼
PREGUNTA ─────→ retrieval
                     │
                     ▼
              TOP-K documentos
                     │
                     ▼
                  contexto
                     │
             pregunta + contexto
                     │
                     ▼
                    LLM
                     │
                     ▼
          respuesta con fuentes
```

Eso es un **RAG (Retrieval-Augmented Generation)** básico.

El objetivo es distinguir con claridad dos subsistemas diferentes:

1. **retrieval**: encontrar la evidencia adecuada;
2. **generation**: utilizar esa evidencia para responder.

## 0. Dependencias

Si vienes siguiendo los notebooks anteriores:

```bash
uv add sentence-transformers scikit-learn pandas openai
```

La API key debe estar fuera del notebook:

```bash
export OPENAI_API_KEY="..."
```

Opcionalmente:

```bash
export OPENAI_MODEL="gpt-6-luna"
```

Usaremos `gpt-6-luna` por defecto para mantener el ejercicio ligero.

## 1. Importaciones y configuración

In [ ]:
import os
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

pd.set_option("display.max_colwidth", 140)

EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
LLM_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "Falta OPENAI_API_KEY. Configúrala como variable de entorno antes de ejecutar el notebook."
    )

client = OpenAI()

print("Embedding model:", EMBEDDING_MODEL)
print("LLM model:", LLM_MODEL)

# 2. Corpus ATM

En un sistema real estos fragmentos podrían proceder de manuales, procedimientos, especificaciones, incidencias o documentación técnica. Aquí utilizamos un corpus pequeño para poder inspeccionarlo visualmente.

In [ ]:
documents = pd.DataFrame([
    {"id": "ATM-001", "area": "capacity",
     "text": "La capacidad del sector se ha reducido por un incremento inesperado del tráfico aéreo."},
    {"id": "ATM-002", "area": "weather",
     "text": "Las tormentas en la zona oeste están obligando a desviar varias rutas y aumentando las demoras."},
    {"id": "ATM-003", "area": "surveillance",
     "text": "Se ha detectado una degradación temporal en la cobertura del radar secundario."},
    {"id": "ATM-004", "area": "communications",
     "text": "La red de comunicaciones tierra-tierra presenta pérdidas intermitentes de conectividad."},
    {"id": "ATM-005", "area": "staffing",
     "text": "La dotación disponible es inferior a la planificada y será necesario reducir la configuración del sector."},
    {"id": "ATM-006", "area": "flight_planning",
     "text": "El sistema de planificación de vuelos está procesando con retraso algunos planes de vuelo."},
    {"id": "ATM-007", "area": "capacity",
     "text": "La demanda prevista supera el número de aeronaves que el sector puede gestionar durante la próxima hora."},
    {"id": "ATM-008", "area": "weather",
     "text": "La presencia de cumulonimbos está generando desvíos y cambios frecuentes de nivel de vuelo."},
    {"id": "ATM-009", "area": "runway",
     "text": "Una de las pistas permanece temporalmente cerrada por inspección y la capacidad del aeropuerto se ha reducido."},
    {"id": "ATM-010", "area": "communications",
     "text": "Se observan dificultades en el intercambio de mensajes entre varios sistemas operacionales."},
    {"id": "ATM-011", "area": "flow_management",
     "text": "Se ha aplicado una regulación ATFM para limitar la entrada de tráfico en el sector congestionado."},
    {"id": "ATM-012", "area": "surveillance",
     "text": "La información de vigilancia presenta actualizaciones irregulares para algunas aeronaves."},
    {"id": "ATM-013", "area": "sector_configuration",
     "text": "Se ha abierto una configuración adicional de sectores para absorber el aumento de demanda."},
    {"id": "ATM-014", "area": "coordination",
     "text": "La carga de coordinación entre sectores adyacentes ha aumentado por múltiples reroutings."},
    {"id": "ATM-015", "area": "normal",
     "text": "Todos los sistemas operacionales funcionan con normalidad y el nivel de tráfico se mantiene estable."},
])

documents

# 3. Indexación: embeddings del corpus

Esta fase suele realizarse antes de que llegue la pregunta del usuario:

```text
documentos → chunking → embeddings → índice vectorial
```

Aquí cada fila ya es un fragmento y almacenaremos sus vectores en memoria.

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

document_embeddings = embedding_model.encode_document(
    documents["text"].tolist(),
    normalize_embeddings=True,
)

print("Embeddings shape:", document_embeddings.shape)

# 4. Retrieval

In [ ]:
def retrieve(question, top_k=3):
    query_embedding = embedding_model.encode_query(
        [question],
        normalize_embeddings=True,
    )

    scores = cosine_similarity(
        query_embedding,
        document_embeddings,
    )[0]

    results = documents.copy()
    results["similarity"] = scores

    return (
        results
        .sort_values("similarity", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

In [ ]:
question = "¿Qué situaciones pueden reducir la capacidad disponible?"
retrieved = retrieve(question, top_k=5)
retrieved.round({"similarity": 3})

Antes de llamar al LLM debemos detenernos aquí. Un RAG tiene **dos sistemas distintos**:

```text
RETRIEVAL  → ¿hemos encontrado la evidencia adecuada?
GENERATION → ¿ha utilizado bien el LLM esa evidencia?
```

Si retrieval falla, un excelente LLM recibirá información equivocada o incompleta.

# 5. Construcción del contexto

In [ ]:
def build_context(retrieved_docs):
    parts = []

    for _, row in retrieved_docs.iterrows():
        parts.append(
            f"[{row['id']}] Área: {row['area']}\n{row['text']}"
        )

    return "\n\n".join(parts)

In [ ]:
context = build_context(retrieved)
print(context)

Incluimos identificadores para que el modelo pueda señalar qué fuente respalda cada afirmación. Esto aporta trazabilidad, aunque una cita generada por el modelo también debe verificarse.

# 6. Generation: responder con la evidencia recuperada

In [ ]:
RAG_INSTRUCTIONS = """
Eres un asistente técnico especializado en ATM.

Reglas:
1. Responde en español.
2. Usa exclusivamente la información del CONTEXTO.
3. No añadas hechos que no estén respaldados por el CONTEXTO.
4. Si el CONTEXTO no permite responder, dilo explícitamente.
5. Para cada afirmación factual relevante, indica la fuente usando su identificador entre corchetes, por ejemplo [ATM-007].
6. No cites una fuente que no aparezca en el CONTEXTO.
"""

def generate_answer(question, context):
    prompt = f"""
CONTEXTO:
{context}

PREGUNTA:
{question}
"""

    response = client.responses.create(
        model=LLM_MODEL,
        instructions=RAG_INSTRUCTIONS,
        input=prompt,
    )

    return response.output_text

In [ ]:
answer = generate_answer(question, context)
print(answer)

# 7. Pipeline RAG completo

In [ ]:
def rag(question, top_k=3, show_retrieval=False):
    retrieved_docs = retrieve(question, top_k=top_k)
    context = build_context(retrieved_docs)
    answer = generate_answer(question, context)

    if show_retrieval:
        return {
            "question": question,
            "retrieved": retrieved_docs,
            "context": context,
            "answer": answer,
        }

    return answer

In [ ]:
result = rag(
    "¿Qué problemas de comunicaciones aparecen en la información disponible?",
    top_k=3,
    show_retrieval=True,
)

print("PREGUNTA")
print(result["question"])

print("\nDOCUMENTOS RECUPERADOS")
display(result["retrieved"].round({"similarity": 3}))

print("\nRESPUESTA")
print(result["answer"])

Acabamos de construir nuestro primer RAG end-to-end:

```text
question → retrieve() → documents → build_context() → context → generate_answer() → answer
```

# 8. Pregunta cuya evidencia está repartida

In [ ]:
question = "¿Qué factores relacionados con meteorología y tráfico pueden afectar a la operación?"

result = rag(question, top_k=5, show_retrieval=True)

display(
    result["retrieved"][["id", "area", "text", "similarity"]]
    .round({"similarity": 3})
)

print(result["answer"])

Aquí aparece uno de los valores de RAG: retrieval encuentra piezas y el LLM puede **sintetizar información procedente de varios fragmentos**.

# 9. Pregunta sin respuesta

In [ ]:
question = "¿Cuál es la frecuencia radio asignada al sector?"

result = rag(question, top_k=3, show_retrieval=True)

display(
    result["retrieved"][["id", "area", "text", "similarity"]]
    .round({"similarity": 3})
)

print(result["answer"])

Esta prueba es crucial: retrieval **siempre devolverá algo**, aunque ningún documento contenga la respuesta.

`top_k=3` significa “devuelve los tres más similares”, no “devuelve tres documentos suficientemente relevantes”.

# 10. Umbral sencillo de retrieval

In [ ]:
def retrieve_with_threshold(question, top_k=3, min_similarity=0.35):
    results = retrieve(question, top_k=top_k)

    return results[
        results["similarity"] >= min_similarity
    ].reset_index(drop=True)

**Advertencia:** `0.35` es solo un valor didáctico. No existe un threshold universal; debe calibrarse con datos de evaluación del dominio.

In [ ]:
question = "¿Cuál es la frecuencia radio asignada al sector?"

retrieve_with_threshold(
    question,
    top_k=5,
    min_similarity=0.35,
).round({"similarity": 3})

In [ ]:
def rag_with_threshold(question, top_k=3, min_similarity=0.35):
    retrieved_docs = retrieve_with_threshold(
        question,
        top_k=top_k,
        min_similarity=min_similarity,
    )

    if retrieved_docs.empty:
        return {
            "retrieved": retrieved_docs,
            "answer": "No se ha encontrado evidencia suficiente en el corpus para responder.",
        }

    context = build_context(retrieved_docs)
    answer = generate_answer(question, context)

    return {
        "retrieved": retrieved_docs,
        "answer": answer,
    }

# 11. Evaluar retrieval sin usar el LLM

In [ ]:
retrieval_eval = [
    {
        "question": "¿Hay problemas de cobertura radar?",
        "expected_ids": {"ATM-003"},
    },
    {
        "question": "¿Qué ocurre con el intercambio de mensajes entre sistemas?",
        "expected_ids": {"ATM-010"},
    },
    {
        "question": "¿La falta de personal puede reducir la configuración?",
        "expected_ids": {"ATM-005"},
    },
    {
        "question": "¿Hay tormentas que obliguen a desviar rutas?",
        "expected_ids": {"ATM-002", "ATM-008"},
    },
    {
        "question": "¿Existe una regulación para limitar tráfico?",
        "expected_ids": {"ATM-011"},
    },
]

In [ ]:
def evaluate_retrieval(eval_set, top_k=3):
    rows = []

    for item in eval_set:
        results = retrieve(item["question"], top_k=top_k)
        retrieved_ids = set(results["id"])
        hit = bool(item["expected_ids"] & retrieved_ids)

        rows.append({
            "question": item["question"],
            "expected": sorted(item["expected_ids"]),
            "retrieved": results["id"].tolist(),
            "hit": hit,
        })

    return pd.DataFrame(rows)

In [ ]:
retrieval_results = evaluate_retrieval(retrieval_eval, top_k=3)
retrieval_results

In [ ]:
hit_rate = retrieval_results["hit"].mean()
print("Hit Rate@3:", round(float(hit_rate), 3))

**Hit Rate@K** responde a una pregunta muy sencilla:

> ¿aparece al menos un documento relevante dentro del top K?

Más adelante podríamos usar Recall@K, Precision@K, MRR, nDCG o evaluaciones específicas del negocio.

# 12. Dos tipos de fallo

### Retrieval correcto, generation incorrecta

El documento adecuado llega al LLM, pero el modelo puede interpretarlo mal, omitir condiciones, mezclar fuentes o añadir información.

### Generation correcta, retrieval insuficiente

El LLM responde perfectamente a lo recibido, pero faltaba una fuente esencial. En ese caso no se arregla cambiando el prompt: hay que mejorar retrieval.

# 13. `top_k` como hiperparámetro del sistema

In [ ]:
question = "¿Qué puede reducir la capacidad?"

for top_k in [1, 2, 3, 5, 8]:
    results = retrieve(question, top_k=top_k)

    print("=" * 80)
    print("TOP_K =", top_k)
    display(
        results[["id", "area", "similarity"]]
        .round({"similarity": 3})
    )

Un `top_k` pequeño puede dejar fuera información relevante. Uno demasiado grande puede introducir ruido, aumentar tokens/coste y distraer al modelo.

# 14. RAG es un patrón arquitectónico

Nuestro RAG usa:

```text
SentenceTransformer
        ↓
cosine similarity
        ↓
top-k
        ↓
prompt
        ↓
OpenAI Responses API
```

Pero cada componente puede sustituirse: búsqueda léxica o híbrida, vector database, reranker, otro modelo de embeddings, otro LLM, structured outputs, etc.

# 15. Vector stores y chunking

Con 15 documentos podemos comparar todos los vectores en memoria. Con millones de fragmentos usaríamos un índice/vector store.

También tendremos que resolver **chunking** cuando trabajemos con documentos reales:

```text
documento grande → chunks → embeddings → índice
```

Chunks demasiado grandes mezclan temas; demasiado pequeños pueden perder contexto.

# 16. Conexión con nuestro futuro proyecto de pruebas

Podemos trasladar ya la arquitectura:

```text
PASO DE PRUEBA EN LENGUAJE NATURAL
              │
              ▼
      recuperar documentación
      ejemplos / restricciones
              │
              ▼
             LLM
              │
              ▼
   interpretar qué acción realizar
              │
              ▼
       TOOL CALLING / MCP
              │
              ▼
        ejecutar acción real
```

RAG aporta **conocimiento**. MCP/tools aportarán **capacidad de actuar**.

# 17. Ejercicios

1. Añade cinco documentos ATM al corpus y recalcula embeddings.
2. Crea dos preguntas cuya respuesta necesite combinar dos documentos.
3. Prueba `top_k = 1`, `3` y `5`.
4. Crea una pregunta que el corpus no pueda responder.
5. Comprueba qué ocurre con y sin threshold.
6. Añade una pregunta y su documento esperado a `retrieval_eval`.
7. Modifica deliberadamente un documento para que contradiga a otro y observa la respuesta.

# 18. Preguntas de comprobación

Al terminar deberías poder explicar:

1. ¿Qué significa RAG?
2. ¿Qué parte corresponde a retrieval?
3. ¿Qué parte corresponde a generation?
4. ¿Por qué calculamos embeddings del corpus antes de recibir preguntas?
5. ¿Qué es `top_k`?
6. ¿Por qué el documento más similar no tiene necesariamente la respuesta?
7. ¿Por qué retrieval y generation deben evaluarse por separado?
8. ¿Qué ocurre si retrieval falla pero el LLM funciona perfectamente?
9. ¿Qué ocurre si retrieval es correcto pero generation falla?
10. ¿Por qué un threshold de similitud no es universal?
11. ¿Qué es Hit Rate@K?
12. ¿Qué papel tiene el chunking?
13. ¿Cuándo necesitamos un vector store?
14. ¿RAG modifica los parámetros del LLM?
15. ¿Qué diferencia existe entre RAG y fine-tuning?
16. ¿Qué aporta RAG al futuro proyecto de pruebas?
17. ¿Qué capacidad falta para que el LLM pueda actuar sobre un sistema real?

# 19. Siguiente paso — Tool Calling

Nuestro recorrido queda así:

```text
ML FUNDAMENTAL       ✓
EMBEDDINGS           ✓
SEMANTIC SEARCH      ✓
LLM                  ✓
PROMPT + CONTEXT     ✓
RAG                  ✓
        │
        ▼
TOOL CALLING
        │
        ▼
MCP
        │
        ▼
AGENTES
```

Ahora el sistema puede **buscar información y responder**. El siguiente paso será hacer que pueda **decidir utilizar una función externa, recibir su resultado y continuar**.